# Limitations of RNNs




## Learning Objectives
---

By the end of this notebook, you should be able to:

- Understand Vanishing and Exploding Gradients mathematically
- Understand Long-Term Dependency challenges in RNN

## Vanishing Gradients
---
RNNs reuse the same weights across all time steps, so the gradients are multiplied repeatedly.

This repeated multiplication can cause them to shrink (vanish) or grow (explode) exponentially, depending on the values.

---
The **vanishing gradient problem** occurs when the backpropagation algorithm moves back through all the neurons of the neural network to update their weights.

<table>
  <tr>
    <td><img src="https://i.postimg.cc/rpXg48qV/VG.png" width="1000"></td>
    <td><img src="https://i.postimg.cc/gJ0hX5Lc/LastUnit.png" width="300"></td>
  </tr>
</table>

As we can see, as the time steps increase, the importance of the initial input decreases further and further. Here, the gradients diminish exponentially as they propagate backward through time, making it difficult to learn connections between events separated by many time steps.

This happens because the RNN repeatedly multiplies derivatives of activation functions (like sigmoid or tanh), which are often less than 1.

Due to Vanishing gradients, early inputs in the sequence barely influence the network’s learning. The RNN struggles to learn long-term dependencies, effectively “forgetting” information from earlier steps.

## Exploding Gradients
---
On the other hand, if the derivatives or weights are larger than 1, the backward signal can grow rapidly as it moves through time, creating the **exploding gradient problem**.

Due to exploding gradients, the network's weights can change too drastically, causing oscillations or instability in training.

---
## Solutions

* Vanishing gradients can be minimized by using gated architectures like LSTM or GRU, which allow the network to preserve information longer.

* Exploding gradients can be minimized by using gradient clipping to limit how large gradients can get during training.


## Mathematical formulas of Vanishing Gradients and Exploding gradients.
---

Here, we will try to understand mathematically about vanishing gradients with the help of equations we had just derived in the previous notebook.


$$ \frac{\partial \mathcal{L}}{\partial W} = \sum_{i=1}^T  \sum_{k=1}^i { \frac{\partial \mathcal{L_i}}{\partial \widehat y_i} } {\frac{\partial \mathcal{\widehat y_i}}{\partial h_i} }   \underbrace{[ \prod_{m = k+1}^i  {\frac{\partial \mathcal{h_m}}{\partial h_{m-1}} }  ]} {\frac{\partial ^+ \mathcal{h_k}}{\partial W} } $$

$$ \frac{\partial \mathcal{L}}{\partial U} = \sum_{i=1}^T \sum_{k=1}^i { \frac{\partial \mathcal{L_i}}{\partial \widehat y_i} } {\frac{\partial \mathcal{\widehat y_i}}{\partial h_i} }   \underbrace{[ \prod_{m = k+1}^i  {\frac{\partial \mathcal{h_m}}{\partial h_{m-1}} }  ]} {\frac{\partial ^+ \mathcal{h_k}}{\partial U} } $$

These two are the equations we had derived earlier. The derivative of Loss with respect to U and W. The actual reason for the vanishing gradient is because of the term that is marked with underbrace.

First of all, let us understand what the relation between $h_m$ and $h_{m-1}$ is

We Know,

$h_m$ = $tanh(U x_m + W h_{m-1})$

Let,

$\phi_j$ = $U x_j + W h_{j-1}$

$h_j$ = $tanh(\phi_j)$


-----
$\underbrace{ \prod_{m = k+1}^i  {\frac{\partial \mathcal{h_m}}{\partial h_{m-1}} }}$

This equation is the product of many terms for every time step. Let us take the jth term from the equation.

$ {\frac{\partial \mathcal{h_j}}{\partial h_{j-1}} }=  {\frac{\partial \mathcal{h_j}}{\partial \phi_j}}  {\frac{\partial \mathcal{\phi_j}}{\partial h_{j-1}} }$

Lets evaluate each of term one by one,

$ {\frac{\partial \mathcal{h_j}}{\partial \phi_j} }$ is,


\begin{bmatrix}
 \  { \frac{\partial \mathcal{h_1}}{\partial \phi_1}} &  { \frac{\partial \mathcal{h_2}}{\partial (\phi_1)}} \hspace{0.4cm} ....  &  { \frac{\partial \mathcal{h_j}}{\partial (\phi_1)}} \\
 { \frac{\partial \mathcal{h_1}}{\partial \phi_2}}   &  { \frac{\partial \mathcal{h_2}}{\partial \phi_2}} \hspace{0.4cm} ....   &  { \frac{\partial \mathcal{h_j}}{\partial \phi_2}}    \\
..... &     &   { \frac{\partial \mathcal{h_j}}{\partial \phi_j}}
\end{bmatrix}

Here, all the non-diagonal elements as 0, so we can write as,


$ {\frac{\partial \mathcal{h_j}}{\partial \phi_j} }= diag(tanh^{'}(\phi_j))$


and the second term,

$ {\frac{\partial \mathcal{\phi_j}}{\partial h_{j-1}} }=  W  $





Now, lets take the magnitude of ,

$ | {\frac{\partial \mathcal{h_j}}{\partial h_{j-1}} } |= | diag(tanh^{'}(\phi_j)) W|$


Using the law of inequality,

$ | {\frac{\partial \mathcal{h_j}}{\partial h_{j-1}} } | <= \underbrace{| diag(tanh^`(\phi_j))|}_1 \underbrace{| W|}_2$

$tanh^`$ is a bounded function with maximum value of (1 = $\alpha$), so $| diag(tanh^`(\phi_j))|$ is bounded by $\alpha$.

$W$ is the weight parameter, which is a real value. Let us suppose it is bounded by $\beta$

Then,

$ | {\frac{\partial \mathcal{h_j}}{\partial h_{j-1}} } | <= (\alpha \beta)$

We are actually interessted in this term, $\underbrace{ \prod_{m = k+1}^i  {\frac{\partial \mathcal{h_m}}{\partial h_{m-1}} }}$

This is the product of many terms.

We can write this as,

$\underbrace{ \prod_{m = k+1}^i  {\frac{\partial \mathcal{h_m}}{\partial h_{m-1}} }}$ $<= \prod_{m=k+1}^i (\alpha \beta)$


$\underbrace{ \prod_{m = k+1}^i  {\frac{\partial \mathcal{h_m}}{\partial h_{m-1}} }}$ $<= (\alpha \beta)^ {i-k}$

* If $( \alpha * \beta )$ > 1, then the series will explode as we are raising to the power to $i-k$, called exploding gradients.

* If $( \alpha * \beta )$ < 1, then the series will vanish as we are raising to the power to $i-k$, called vanishing gradients.

*Note: Here, $i-k$ refers backpropagating from ith timestep to kth timestep.*






## Long-term dependencies
---

As we previously learned, long-term dependencies occur when the prediction at the current time step depends on inputs from far earlier in the sequence.


Vanilla RNNs struggle with long-term dependencies due to vanishing gradients.

RNNs update the hidden state

$$ℎ_𝑡 = 𝑓(𝑥_𝑡, ℎ_{𝑡−1})$$

step by step.

As sequences get longer, the influence of $ℎ_0$ (initial input) on later hidden states diminishes, as its value shrinks exponentially.

Thus, the network remembers only short-term context reliably, and fails to capture long-range dependencies, which are crucial in many real-world sequence tasks like:
- Language modeling
- Speech recognition
- Long time-series forecasting



### Solutions / Workarounds
---

- **LSTM (Long Short-Term Memory)**: introduces gates (input, forget, output) to preserve information over longer sequences.

- **GRU (Gated Recurrent Unit)**: simpler variant of LSTM, also mitigates vanishing gradients.

- **Attention / Transformers**: bypass recurrence entirely, allowing direct connections between all positions in the sequence.

We will cover these topics in the next lessons.